In [1]:
%run shared_imports.py
from decimal import Decimal
from dataclasses import dataclass
from sqlalchemy.orm.query import Query


In [2]:
engine = make_engine("settings.toml")
session = Session(engine)


In [55]:
query = session.query(Round.id, Round.game_mode, Feedback.json).join(Round).filter(
    Feedback.key_name == "ore_mined", 
    Round.start_datetime >= datetime(2023, 10, 5))

raw_ore_mined = pd.read_sql_query(query.statement, session.connection())

In [56]:
raw_ore_mined

,id,game_mode,json
0,37561,wizard,"{'data': {'/obj/item/stack/ore/iron': 913, '/o..."
1,37562,wizard,"{'data': {'/obj/item/stack/ore/iron': 1224, '/..."
2,37563,nuclear emergency,"{'data': {'/obj/item/stack/ore/gold': 115, '/o..."
3,37564,extended,"{'data': {'/obj/item/stack/ore/iron': 1808, '/..."
4,37565,traitor_vampire,"{'data': {'/obj/item/stack/ore/uranium': 256, ..."
...,...,...,...
4463,42066,traitor_vampire,"{'data': {'/obj/item/stack/ore/iron': 1075, '/..."
4464,42067,vampire_changeling,"{'data': {'/obj/item/stack/ore/titanium': 370,..."
4465,42068,Trifecta,"{'data': {'/obj/item/stack/ore/iron': 1258, '/..."
4466,42069,ragin' mages,"{'data': {'/obj/item/stack/ore/iron': 331, '/o..."


In [57]:
normalized_ore_mined

,data./obj/item/stack/ore/uranium,data./obj/item/stack/ore/iron,data./obj/item/stack/ore/plasma,data./obj/item/stack/ore/gold,data./obj/item/stack/ore/silver,data./obj/item/stack/ore/titanium,data./obj/item/stack/ore/bluespace_crystal,data./obj/item/stack/ore/diamond,data./obj/item/stack/ore/bananium,data./obj/item/stack/ore/tranquillite,data./obj/item/stack/ore/glass/basalt/ancient
0,242.0,2936.0,1146.0,552.0,625.0,492.0,59.0,114.0,NaN,NaN,NaN
1,335.0,3468.0,1215.0,524.0,663.0,656.0,54.0,36.0,NaN,NaN,NaN
2,253.0,2340.0,839.0,388.0,457.0,467.0,39.0,49.0,NaN,NaN,NaN
3,389.0,2568.0,1276.0,604.0,646.0,727.0,62.0,104.0,NaN,NaN,NaN
4,67.0,213.0,155.0,126.0,110.0,82.0,8.0,16.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16605,72.0,1075.0,306.0,161.0,214.0,146.0,29.0,34.0,NaN,NaN,812.0
16606,232.0,2018.0,862.0,302.0,441.0,370.0,19.0,38.0,NaN,NaN,6.0
16607,126.0,1258.0,612.0,300.0,323.0,280.0,23.0,33.0,NaN,NaN,28.0
16608,49.0,331.0,206.0,100.0,106.0,84.0,11.0,17.0,NaN,NaN,NaN


In [58]:
normalized = pd.concat([raw_ore_mined, raw_ore_mined.json.apply(pd.json_normalize)]).drop(['json'], inplace=True, axis=1)

In [22]:
normalized.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [59]:
concatted = pd.concat([raw_ore_mined, pd.json_normalize(raw_ore_mined.json)], axis=1).drop(['json'], axis=1)

In [60]:
concatted.columns = concatted.columns.str.replace("data./obj/item/stack/ore/", "", regex=True)

In [61]:
concatted

,id,game_mode,iron,gold,plasma,titanium,silver,uranium,bluespace_crystal,diamond,glass/basalt/ancient,bananium,tranquillite
0,37561,wizard,913.0,103.0,320.0,143.0,124.0,42.0,5.0,5.0,NaN,NaN,NaN
1,37562,wizard,1224.0,213.0,532.0,194.0,239.0,129.0,18.0,29.0,194.0,NaN,NaN
2,37563,nuclear emergency,483.0,115.0,228.0,82.0,104.0,47.0,25.0,6.0,NaN,NaN,NaN
3,37564,extended,1808.0,265.0,659.0,206.0,329.0,162.0,52.0,33.0,NaN,NaN,NaN
4,37565,traitor_vampire,2260.0,391.0,850.0,376.0,468.0,256.0,66.0,77.0,348.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4463,42066,traitor_vampire,1075.0,161.0,306.0,146.0,214.0,72.0,29.0,34.0,812.0,NaN,NaN
4464,42067,vampire_changeling,2018.0,302.0,862.0,370.0,441.0,232.0,19.0,38.0,6.0,NaN,NaN
4465,42068,Trifecta,1258.0,300.0,612.0,280.0,323.0,126.0,23.0,33.0,28.0,NaN,NaN
4466,42069,ragin' mages,331.0,100.0,206.0,84.0,106.0,49.0,11.0,17.0,NaN,NaN,NaN


In [62]:
concatted[(concatted.iron < 500) & (concatted.game_mode.isin(["wizard", "ragin' mages", "nuclear emergency", "blob"]))]

,id,game_mode,iron,gold,plasma,titanium,silver,uranium,bluespace_crystal,diamond,glass/basalt/ancient,bananium,tranquillite
2,37563,nuclear emergency,483.0,115.0,228.0,82.0,104.0,47.0,25.0,6.0,NaN,NaN,NaN
42,37603,wizard,343.0,90.0,204.0,98.0,70.0,66.0,7.0,29.0,2.0,NaN,NaN
62,37623,wizard,446.0,89.0,82.0,72.0,33.0,40.0,13.0,9.0,NaN,NaN,NaN
71,37632,wizard,355.0,126.0,120.0,103.0,54.0,72.0,30.0,15.0,62.0,NaN,NaN
97,37658,wizard,471.0,67.0,200.0,138.0,87.0,56.0,NaN,11.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4428,42031,wizard,196.0,37.0,91.0,48.0,84.0,30.0,13.0,25.0,NaN,NaN,NaN
4429,42032,nuclear emergency,38.0,12.0,32.0,11.0,9.0,1.0,NaN,NaN,NaN,NaN,NaN
4431,42034,wizard,125.0,67.0,66.0,40.0,63.0,24.0,14.0,17.0,102.0,NaN,NaN
4457,42060,nuclear emergency,424.0,98.0,275.0,96.0,93.0,84.0,12.0,22.0,4.0,NaN,NaN


In [63]:
concatted[(concatted.iron < 500)]

,id,game_mode,iron,gold,plasma,titanium,silver,uranium,bluespace_crystal,diamond,glass/basalt/ancient,bananium,tranquillite
2,37563,nuclear emergency,483.0,115.0,228.0,82.0,104.0,47.0,25.0,6.0,NaN,NaN,NaN
41,37602,extended,121.0,21.0,93.0,56.0,54.0,15.0,8.0,3.0,NaN,NaN,NaN
42,37603,wizard,343.0,90.0,204.0,98.0,70.0,66.0,7.0,29.0,2.0,NaN,NaN
62,37623,wizard,446.0,89.0,82.0,72.0,33.0,40.0,13.0,9.0,NaN,NaN,NaN
68,37629,AutoTraitor,270.0,96.0,88.0,161.0,136.0,99.0,10.0,7.0,138.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4441,42044,vampire,349.0,145.0,189.0,115.0,154.0,64.0,11.0,21.0,NaN,NaN,NaN
4448,42051,extended,400.0,71.0,154.0,77.0,68.0,57.0,3.0,20.0,NaN,NaN,NaN
4452,42055,traitor,297.0,48.0,292.0,91.0,45.0,39.0,7.0,7.0,NaN,NaN,NaN
4457,42060,nuclear emergency,424.0,98.0,275.0,96.0,93.0,84.0,12.0,22.0,4.0,NaN,NaN


In [64]:
concatted.game_mode.unique()

array(['wizard', 'nuclear emergency', 'extended', 'traitor_vampire',
       'traitor', 'vampire', 'AutoTraitor', 'cult', 'Trifecta',
       'traitor_changeling', 'changeling', 'revolution', "ragin' mages",
       'abduction', None, 'vampire_changeling'], dtype=object)

In [66]:
concatted.sum(numeric_only=True)

id                      177889886.0
iron                      7850211.0
gold                      1286213.0
plasma                    2853621.0
titanium                  1304484.0
silver                    1424891.0
uranium                    714460.0
bluespace_crystal          156140.0
diamond                    160236.0
glass/basalt/ancient       278837.0
bananium                     4371.0
tranquillite                  115.0
dtype: float64